In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
emotion_df = pd.read_csv("final_emotion_dataset.csv")
emotion_df.head()

,text,sentiment,clean_text,label,main_emotion,main_label
0,I experienced this emotion when my grandfather...,sadness,experienced emotion grandfather passed away,26,Sad,8
1,"when I first moved in , I walked everywhere ....",neutral,first moved walked everywhere within week purs...,20,Neutral,6
2,"` Oh ! "" she bleated , her voice high and rath...",anger,oh bleated voice high rather indignant,2,Angry,1
3,"However , does the right hon. Gentleman recogn...",fear,however right hon gentleman recognise profound...,14,Fear,4
4,My boyfriend didn't turn up after promising th...,sadness,boyfriend not turn promising coming,26,Sad,8


In [5]:
print(emotion_df.shape)
print(emotion_df.columns)
print(emotion_df["sentiment"].nunique())   

(200235, 6)
Index(['text', 'sentiment', 'clean_text', 'label', 'main_emotion',
       'main_label'],
      dtype='object')
28


In [6]:
emotion_df["clean_text"] = emotion_df["clean_text"].fillna("")
emotion_df["text"] = emotion_df["text"].fillna("")

In [7]:
X = emotion_df["clean_text"]
y = emotion_df["main_label"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (160188,)
Testing : (40047,)


In [10]:
print(X_train.isna().sum())
print(X_test.isna().sum())

0
0


In [11]:
print(emotion_df["clean_text"].isna().sum())

0


In [12]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)


(160188, 50000)
(40047, 50000)


In [13]:
X_train_embed = model.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_embed = model.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/2503 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_matrix = cosine_similarity(X_train_tfidf[:1000])

print(cosine_matrix.shape)

(1000, 1000)


In [72]:
lr_tfidf = LogisticRegression(max_iter=1000)

lr_tfidf.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [73]:
y_pred_lr = lr_tfidf.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.500536869178715
              precision    recall  f1-score   support

           0       0.21      0.04      0.07      3138
           1       0.39      0.34      0.36      5289
           2       0.29      0.06      0.10      2019
           3       0.40      0.01      0.01       283
           4       0.68      0.38      0.49      3086
           5       0.53      0.59      0.56      7819
           6       0.40      0.63      0.49     10667
           7       0.00      0.00      0.00       141
           8       0.77      0.72      0.74      7605

    accuracy                           0.50     40047
   macro avg       0.41      0.31      0.31     40047
weighted avg       0.49      0.50      0.48     40047



c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [74]:
lr_sbert = LogisticRegression(
    max_iter=2000,
    random_state=42
)
lr_sbert.fit(X_train_embed, y_train)
y_pred = lr_sbert.predict(X_test_embed)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.4917222263839988


In [ ]:
svm = LinearSVC()

svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))

Accuracy: 0.4588358678552701


In [ ]:
nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))

Accuracy: 0.4618573176517592


In [48]:
print(emotion_df["sentiment"].value_counts())

sentiment
neutral           48949
sadness           20281
admiration        12377
approval          11882
grief             11053
annoyance          9128
anger              7761
fear               7493
disapproval        6829
joy                6219
disappointment     5492
curiosity          5204
amusement          5015
confusion          4816
gratitude          4441
realization        4386
optimism           3837
caring             3809
excitement         3441
nervousness        3120
love               3117
disgust            2728
surprise           2604
desire             2286
embarrassment      1415
remorse            1197
relief              706
pride               649
Name: count, dtype: int64


In [49]:
print("Total emotions:", emotion_df["sentiment"].nunique())

Total emotions: 28


In [50]:
print(emotion_df.columns)
print()
print(emotion_df.sample(20)[["text", "sentiment"]])
print()
print(emotion_df["clean_text"].head(20))

Index(['text', 'sentiment', 'clean_text', 'label', 'main_emotion',
       'main_label'],
      dtype='object')

                                                     text       sentiment
75873   Agree 100% I guess I'm just happy to see anyon...         neutral
118213             It’s objectively a lie. Prove me wrong           anger
50200   I do not understand why everything in this scr...         disgust
91878   I didn't downvote you so I couldn't tell yah w...         neutral
63306   If that's what you call ugly, I'm basically a ...     disapproval
194833               feel like cry that s how sick i feel         neutral
45247   This would make me sick.Getting cucked by baye...         neutral
118109                In a sous vide? How does that work?       confusion
40747   Oh I gets it. Sort of wished these two guys we...        surprise
81577   This would still happen if they didn't have ki...        approval
136601                                    This is amazing             joy


In [51]:
print(X_train.shape)
print(X_test.shape)
print(y_train.nunique())
print(y_test.nunique())
print(tfidf.vocabulary_.__len__())

(160188,)
(40047,)
9
9
50000


In [ ]:
print(emotion_df["clean_text"].sample(20).tolist())

['thanks book recommendation', 'mean everyone already know u like gonna weird', 'we’re still commenting video dog farting cat throwing right', 'stuck long time cannot see get better never get closure justice people hurt never going acknowledge believe good people everyone agrees accept unfairness let go move forward without closure justice', 'yes however misquote study however ive corrected op', 'not child yet long conversation ball tonight', 'making photo look pretty', 'wearing little sister jean bc stole juice thats', 'seems full also seems like ok dude', 'want sheet without stain', 'retarded', 'that’s shame what’s worse evangelicalism scientology tough call', 'heart attack scare anxiety roof hi posted ranxiety well thought id post hope thats ok friday night rushed emergency department severe chest pain point could not even talk cry not know could didnt even cross mind could anything heart pain eased hour around hour finally got seen doctor pain non existent explained issue doctor to

In [54]:
print("label_encoder" in globals())
print("main_label_encoder" in globals())

False
False


In [17]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
emotion_df["label"] = label_encoder.fit_transform(emotion_df["sentiment"])

main_label_encoder = LabelEncoder()
emotion_df["main_label"] = main_label_encoder.fit_transform(emotion_df["main_emotion"])

In [56]:
joblib.dump(label_encoder, "../models/sub_emotion_encoder.pkl")
joblib.dump(main_label_encoder, "../models/main_emotion_encoder.pkl")

['../models/main_emotion_encoder.pkl']

In [58]:
mapping = (
    emotion_df.groupby("main_emotion")["sentiment"]
    .unique()
    .apply(list)
    .to_dict()
)

joblib.dump(mapping, "../models/main_to_sub_mapping.pkl")

['../models/main_to_sub_mapping.pkl']

In [59]:
model = joblib.load("../models/main_emotion_model.pkl")
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

print("Everything loaded successfully.")

Everything loaded successfully.


In [75]:
print(model.n_features_in_)

384


In [79]:
joblib.dump(lr_tfidf, "../models/main_emotion_model_tfidf.pkl")


['../models/main_emotion_model_tfidf.pkl']

In [80]:
joblib.dump(lr_sbert, "../models/main_emotion_model_sbert.pkl")

['../models/main_emotion_model_sbert.pkl']

In [82]:
import joblib

vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")
model = joblib.load("../models/main_emotion_model_tfidf.pkl")
main_label_encoder = joblib.load("../models/main_emotion_encoder.pkl")

In [83]:
print(model.n_features_in_)

50000


In [84]:
text = "I feel very lonely and hopeless."

x = vectorizer.transform([text])
pred = model.predict(x)[0]
main = main_label_encoder.inverse_transform([pred])[0]

print(main)

Sad


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# 28 emotion labels
y_train_sub = emotion_df.loc[X_train.index, "label"]
y_test_sub = emotion_df.loc[X_test.index, "label"]

sub_model = LogisticRegression(
    max_iter=2000,
    random_state=42,
    n_jobs=-1
)

sub_model.fit(X_train_tfidf, y_train_sub)
pred_sub = sub_model.predict(X_test_tfidf)
print("Sub Emotion Accuracy:", accuracy_score(y_test_sub, pred_sub))

print(classification_report(
    y_test_sub,
    pred_sub,
    target_names=label_encoder.classes_
))

Sub Emotion Accuracy: 0.37283691662296803
                precision    recall  f1-score   support

    admiration       0.29      0.23      0.25      2449
     amusement       0.34      0.24      0.28      1074
         anger       0.34      0.17      0.23      1503
     annoyance       0.10      0.03      0.05      1827
      approval       0.14      0.04      0.06      2360
        caring       0.18      0.03      0.06       778
     confusion       0.26      0.03      0.06       955
     curiosity       0.23      0.02      0.04      1055
        desire       0.27      0.07      0.11       452
disappointment       0.12      0.01      0.02      1116
   disapproval       0.13      0.03      0.05      1405
       disgust       0.20      0.06      0.09       554
 embarrassment       0.25      0.01      0.02       283
    excitement       0.15      0.03      0.05       657
          fear       0.66      0.51      0.58      1507
     gratitude       0.44      0.43      0.44       893
     

In [19]:
import joblib
joblib.dump(
    sub_model,
    "../models/sub_emotion_model_tfidf.pkl"
)

print("Sub Emotion model saved successfully.")

Sub Emotion model saved successfully.


In [20]:
text = "I feel lonely and hopeless after failing my exam."

x = tfidf.transform([text])
pred = sub_model.predict(x)[0]
sub_emotion = label_encoder.inverse_transform([pred])[0]

print("Predicted Sub Emotion:", sub_emotion)

Predicted Sub Emotion: sadness
